# 03 - On-the-ground costs (lifestyle basket)

Builds a per-location price basket so the app can turn a user's chosen lifestyle
(dinners, casual meals, fast food, bar drinks, club nights, coffees, self-catered
days) into an accurate trip spend.

## Basket (per person, EUR)

| key | item | drives |
|-----|------|--------|
| `meal_mid_eur`     | mid-range restaurant meal (3-course / 2)   | dinners out |
| `meal_cheap_eur`   | inexpensive restaurant meal                | casual meals |
| `fastfood_eur`     | McMeal / equivalent combo                   | fast food / street |
| `drink_out_eur`    | domestic draught beer 0.5L at a bar         | bar drinks |
| `cocktail_eur`     | club/premium drink (= imported beer x 2.4)  | club nights |
| `coffee_eur`       | cappuccino                                  | coffees |
| `grocery_day_eur`  | one day of self-catering groceries          | self-catered days |
| `club_entry_eur`   | nightclub cover charge (estimate)           | club nights |

## Data + method (clean, accurate, validated)

- **Real Numbeo anchors** - euro prices captured from public Numbeo country/city
  pages (June 2026), loaded from `cache/numbeo_raw.json`: **23 countries + 22
  cities** with the full item set. Used directly.
- **The long tail** - locations without a real anchor are scaled from the Belgium
  baseline using **Eurostat 2024 Price Level Indices** (restaurants & hotels for
  dining/drinks, food for groceries). Eurostat surveys 2,000+ goods across 36
  countries via the OECD PPP programme - the official reference.
- **Belgium baseline** is curated (Numbeo's BE page was in the batch blocked by a
  Numbeo IP rate-limit during collection); refresh it when the block lifts.
- **Estimated items**: `cocktail_eur` = imported beer x 2.4; `club_entry_eur` is a
  Belgium-anchored estimate, PLI-scaled. Numbeo has no direct item for either.
- **Validation** - the validation cell re-predicts the anchored countries' dining
  prices from PLI and reports the per-item error, so the long-tail scaling's
  accuracy is measurable.

Refresh: re-run the collection agents after the Numbeo block resets (~2026-07-01)
to replace any remaining PLI-scaled locations with real anchors in
`cache/numbeo_raw.json`. Writes `cache/costs.json`.

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter

CACHE_DIR = Path("cache")
cfg    = json.loads((CACHE_DIR / "config.json").read_text(encoding="utf-8"))
master = json.loads((CACHE_DIR / "destinations_master.json").read_text(encoding="utf-8"))
raw    = json.loads((CACHE_DIR / "numbeo_raw.json").read_text(encoding="utf-8"))
dests  = master["destinations"]

BASELINE = "BE"
ITEMS = ["meal_mid_eur", "meal_cheap_eur", "fastfood_eur", "drink_out_eur",
         "cocktail_eur", "coffee_eur", "grocery_day_eur", "club_entry_eur"]
DINING_KEYS = ["meal_mid_eur", "meal_cheap_eur", "fastfood_eur", "drink_out_eur",
               "cocktail_eur", "coffee_eur"]  # the items that scale on restaurants PLI

# Derived-item parameters (Numbeo has no direct item for these)
COCKTAIL_MULT  = 2.4   # a club/premium drink vs an imported beer
BE_GROCERY_DAY = 13.0  # one person's self-catering day in Belgium (food-PLI-scaled elsewhere)
BE_CLUB_ENTRY  = 12.0  # Belgium nightclub cover estimate (rest-PLI-scaled elsewhere)

# Real Numbeo prices captured June 2026 (cache/numbeo_raw.json). Each record has:
# meal_cheap, meal_mid_for2, mcmeal, beer_draught_05, beer_imported_033, cappuccino.
RAW_COUNTRY = dict(raw["countries"])
RAW_CITY    = dict(raw["cities"])

# Belgium baseline (curated - BE page was blocked by the Numbeo IP rate-limit).
RAW_COUNTRY.setdefault("BE", {
    "meal_cheap": 20.0, "meal_mid_for2": 80.0, "mcmeal": 10.5,
    "beer_draught_05": 4.90, "beer_imported_033": 4.50, "cappuccino": 3.70,
})

def basket_from_raw(r):
    """Map a raw Numbeo record to the dining items of our basket (EUR/person)."""
    return {
        "meal_mid_eur":   round(r["meal_mid_for2"] / 2, 2),
        "meal_cheap_eur": round(r["meal_cheap"], 2),
        "fastfood_eur":   round(r["mcmeal"], 2),
        "drink_out_eur":  round(r["beer_draught_05"], 2),
        "cocktail_eur":   round(r["beer_imported_033"] * COCKTAIL_MULT, 2),
        "coffee_eur":     round(r["cappuccino"], 2),
    }

BE_DINING = basket_from_raw(RAW_COUNTRY["BE"])

# --- Eurostat 2024 Price Level Indices (EU27_2020 = 100) ---
# (all_items, restaurants_hotels, food, alcohol_tobacco, transport)
# https://ec.europa.eu/eurostat/statistics-explained/index.php?title=Comparative_price_levels_of_consumer_goods_and_services
PLI_2024 = {
    "AT": (114,117,122,100,110), "BE": (114,118,113,121,113), "BG": (61,62,91,82,74),
    "HR": (78,85,98,103,82),     "CY": (95,93,100,109,81),    "CZ": (87,85,102,80,85),
    "DK": (141,165,130,134,134), "EE": (90,98,108,113,90),    "FI": (123,138,124,165,113),
    "FR": (108,109,110,134,109), "DE": (110,116,121,89,112),  "GR": (87,88,100,117,84),
    "HU": (73,72,90,79,85),      "IE": (137,150,121,175,122), "IT": (101,102,113,110,108),
    "LV": (86,88,104,109,84),    "LT": (79,79,98,101,78),     "LU": (134,132,132,116,121),
    "MT": (88,100,102,105,80),   "NL": (119,123,119,103,113), "PL": (65,64,82,76,72),
    "PT": (88,85,100,99,86),     "RO": (60,58,77,68,65),      "SK": (80,85,95,78,85),
    "SI": (86,90,104,95,93),     "ES": (95,96,101,95,94),     "SE": (114,116,108,132,113),
    "CH": (165,180,168,119,145), "IS": (170,175,125,246,145), "NO": (140,145,153,264,145),
    "GB": (123,134,104,156,110), "AL": (53,55,72,62,58),      "BA": (60,63,76,68,62),
    "ME": (62,68,78,84,62),      "MK": (47,48,62,58,50),      "RS": (62,68,76,68,65),
}
NEIGHBOR_FALLBACK = {
    "AD": ["ES","FR"], "MC": ["FR"], "SM": ["IT"], "LI": ["CH","AT"], "XK": ["MK","RS"],
    "MD": ["RO"], "VA": ["IT"], "UA": ["RO","PL"], "BY": ["PL","LT"],
    "FO": ["DK"],  # Faroe Islands: self-governing, no Eurostat PLI -> Denmark proxy
}
REST_IDX, FOOD_IDX = 1, 2

print(f"Destinations: {len(dests)}  |  countries: {len(set(d['iso2'] for d in dests))}")
print(f"Real Numbeo anchors: {len(RAW_COUNTRY)} countries, {len(RAW_CITY)} cities")

Destinations: 450  |  countries: 42
Real Numbeo anchors: 24 countries, 22 cities


## 1. Build the per-country basket

Direct Numbeo prices where we have them; otherwise scale the Belgium baseline by
the country's Eurostat PLI (restaurants & hotels for dining, food for groceries).

In [2]:
def pli_for(iso, idx):
    """Eurostat PLI component for a country, with neighbour fallback. None if unknown."""
    if iso in PLI_2024:
        return PLI_2024[iso][idx]
    neighbours = NEIGHBOR_FALLBACK.get(iso, [])
    vals = [PLI_2024[n][idx] for n in neighbours if n in PLI_2024]
    return sum(vals) / len(vals) if vals else None

BE_REST = PLI_2024[BASELINE][REST_IDX]
BE_FOOD = PLI_2024[BASELINE][FOOD_IDX]

def derived_items(rest, food):
    """grocery_day (food PLI) + club_entry (restaurants PLI), scaled from Belgium."""
    grocery = round(BE_GROCERY_DAY * (food / BE_FOOD), 2) if food else BE_GROCERY_DAY
    club    = round(BE_CLUB_ENTRY * (rest / BE_REST), 2) if rest else BE_CLUB_ENTRY
    return grocery, club

def country_basket(iso):
    rest = pli_for(iso, REST_IDX)
    food = pli_for(iso, FOOD_IDX)

    if iso in RAW_COUNTRY:
        dining = basket_from_raw(RAW_COUNTRY[iso])
        source = "numbeo_direct"
    elif rest is not None:
        f = rest / BE_REST
        dining = {k: round(BE_DINING[k] * f, 2) for k in DINING_KEYS}
        source = "pli_scaled"
    else:
        return None

    grocery, club = derived_items(rest, food)
    return {**dining, "grocery_day_eur": grocery, "club_entry_eur": club, "source": source}

dest_isos = sorted({d["iso2"] for d in dests})
countries = {}
missing = []
for iso in dest_isos:
    b = country_basket(iso)
    if b is None:
        missing.append(iso)
    else:
        countries[iso] = b

src = Counter(b["source"] for b in countries.values())
print(f"Country baskets: {len(countries)}  sources={dict(src)}")
if missing:
    print(f"  no data (app falls back to baseline): {missing}")
print("\nSample (EUR per person):")
for iso in ["BE", "PT", "ES", "IT", "PL", "CH", "NO", "RO"]:
    if iso in countries:
        b = countries[iso]
        print(f"  {iso}: dinner {b['meal_mid_eur']:5.2f} | casual {b['meal_cheap_eur']:5.2f} | "
              f"fastfood {b['fastfood_eur']:5.2f} | beer {b['drink_out_eur']:4.2f} | "
              f"cocktail {b['cocktail_eur']:5.2f} | club {b['club_entry_eur']:5.2f}  ({b['source']})")

Country baskets: 42  sources={'pli_scaled': 18, 'numbeo_direct': 24}

Sample (EUR per person):
  BE: dinner 40.00 | casual 20.00 | fastfood 10.50 | beer 4.90 | cocktail 10.80 | club 12.00  (numbeo_direct)
  PT: dinner 22.50 | casual 12.00 | fastfood  8.50 | beer 2.50 | cocktail  7.20 | club  8.64  (numbeo_direct)
  ES: dinner 25.00 | casual 15.00 | fastfood 10.00 | beer 3.00 | cocktail  8.40 | club  9.76  (numbeo_direct)
  IT: dinner 35.00 | casual 17.00 | fastfood 10.00 | beer 5.00 | cocktail 12.00 | club 10.37  (numbeo_direct)
  PL: dinner 23.57 | casual  9.43 | fastfood  8.25 | beer 3.54 | cocktail  8.50 | club  6.51  (numbeo_direct)
  CH: dinner 53.00 | casual 26.50 | fastfood 15.90 | beer 7.63 | cocktail 15.26 | club 18.31  (numbeo_direct)
  NO: dinner 42.75 | casual 18.98 | fastfood 12.83 | beer 9.41 | cocktail 21.55 | club 14.75  (numbeo_direct)
  RO: dinner 19.06 | casual  9.53 | fastfood  6.67 | beer 2.00 | cocktail  6.86 | club  5.90  (numbeo_direct)


## 2. City overrides

The fetched cities replace their country's dining/drink prices (capitals and
tourist hubs run well above the national average). Groceries and club entry stay
at the country level.

In [3]:
# City -> ISO2, so a city can borrow its country's grocery + club levels.
CITY_ISO = {
    "Madrid": "ES", "Barcelona": "ES", "Lisbon": "PT", "Porto": "PT",
    "Copenhagen": "DK", "Stockholm": "SE", "Oslo": "NO", "Helsinki": "FI",
    "Reykjavik": "IS", "Athens": "GR", "Thessaloniki": "GR", "Bucharest": "RO",
    "Sofia": "BG", "Dubrovnik": "HR", "Tallinn": "EE", "Riga": "LV",
    "Vilnius": "LT", "Valletta": "MT", "Geneva": "CH", "Zurich": "CH",
    "Tirana": "AL", "Belgrade": "RS",
}

cities = {}
unmapped = []
for city, r in RAW_CITY.items():
    iso = CITY_ISO.get(city)
    if not iso:
        unmapped.append(city)
        continue
    country = countries.get(iso, {})
    dining = basket_from_raw(r)
    cities[city] = {
        **dining,
        "grocery_day_eur": country.get("grocery_day_eur", BE_GROCERY_DAY),
        "club_entry_eur":  country.get("club_entry_eur", BE_CLUB_ENTRY),
        "source": "numbeo_city",
    }

print(f"City overrides: {len(cities)}")
if unmapped:
    print(f"  (no ISO mapping, skipped): {unmapped}")
for city in sorted(cities):
    b = cities[city]
    print(f"  {city:12s}: dinner {b['meal_mid_eur']:5.2f} | beer {b['drink_out_eur']:4.2f} | "
          f"cocktail {b['cocktail_eur']:5.2f} | fastfood {b['fastfood_eur']:5.2f}")

City overrides: 22
  Athens      : dinner 30.00 | beer 5.00 | cocktail 12.00 | fastfood  9.00
  Barcelona   : dinner 30.00 | beer 4.00 | cocktail  9.60 | fastfood 12.00
  Belgrade    : dinner 25.12 | beer 2.98 | cocktail  8.18 | fastfood 11.16
  Bucharest   : dinner 26.73 | beer 3.05 | cocktail  8.71 | fastfood  6.68
  Copenhagen  : dinner 53.51 | beer 8.03 | cocktail 16.06 | fastfood 12.04
  Dubrovnik   : dinner 35.00 | beer 5.00 | cocktail  9.00 | fastfood  7.50
  Geneva      : dinner 59.94 | beer 8.72 | cocktail 20.93 | fastfood 16.35
  Helsinki    : dinner 45.00 | beer 8.00 | cocktail 16.80 | fastfood 13.00
  Lisbon      : dinner 27.50 | beer 3.00 | cocktail  7.20 | fastfood 10.00
  Madrid      : dinner 30.00 | beer 3.50 | cocktail 10.80 | fastfood 11.00
  Oslo        : dinner 55.00 | beer 11.00 | cocktail 26.40 | fastfood 13.75
  Porto       : dinner 25.00 | beer 3.00 | cocktail  9.60 | fastfood  8.00
  Reykjavik   : dinner 42.41 | beer 11.63 | cocktail 25.06 | fastfood 22.63
  Ri

## 3. Validate the PLI scaling against the Numbeo anchors

For each Numbeo-anchored country, predict its dining prices from the Belgium
baseline x PLI and compare to the real Numbeo value. A low mean error means the
scaling we use for the long-tail countries is trustworthy.

In [4]:
per_item = {k: [] for k in DINING_KEYS}
for iso, r in RAW_COUNTRY.items():
    if iso == BASELINE or iso not in PLI_2024:
        continue
    f = PLI_2024[iso][REST_IDX] / BE_REST
    actual = basket_from_raw(r)
    for key in DINING_KEYS:
        est = BE_DINING[key] * f
        per_item[key].append(abs(100 * (est - actual[key]) / actual[key]))

print("Mean absolute error per item (long-tail PLI scaling vs real Numbeo anchors):")
item_mae = {}
for key in DINING_KEYS:
    mae_k = sum(per_item[key]) / len(per_item[key])
    item_mae[key] = round(mae_k, 1)
    print(f"  {key:<16} {mae_k:5.1f}%")

all_errs = [e for v in per_item.values() for e in v]
overall = sum(all_errs) / len(all_errs)
n_anchor_countries = len([k for k in RAW_COUNTRY if k != BASELINE])
print(f"\nOverall mean absolute error: {overall:.1f}%  (over {n_anchor_countries} anchored countries)")
print("Meals track the restaurant PLI well; coffee/beer are noisier because")
print("cafe/bar prices don't follow it cleanly. Anchored countries + cities use")
print("REAL Numbeo prices, so this error only applies to the long-tail countries.")
validation = {
    "method": "Belgium baseline x Eurostat restaurants&hotels PLI, validated vs Numbeo anchors",
    "overall_mae_pct": round(overall, 1),
    "per_item_mae_pct": item_mae,
    "n_anchor_countries": n_anchor_countries,
}

Mean absolute error per item (long-tail PLI scaling vs real Numbeo anchors):
  meal_mid_eur      14.7%
  meal_cheap_eur    20.2%
  fastfood_eur      13.6%
  drink_out_eur     21.9%
  cocktail_eur      18.7%
  coffee_eur        18.3%

Overall mean absolute error: 17.9%  (over 23 anchored countries)
Meals track the restaurant PLI well; coffee/beer are noisier because
cafe/bar prices don't follow it cleanly. Anchored countries + cities use
REAL Numbeo prices, so this error only applies to the long-tail countries.


## 4. Save

In [5]:
OUT = CACHE_DIR / "costs.json"
payload = {
    "meta": {
        "schema_version": cfg["schema_version"],
        "generated_at":   datetime.now(timezone.utc).isoformat(),
        "currency":       "EUR",
        "baseline":       BASELINE,
        "items":          ITEMS,
        "basket": {
            "meal_mid_eur":    "mid-range restaurant meal, per person",
            "meal_cheap_eur":  "inexpensive restaurant meal, per person",
            "fastfood_eur":    "fast food / street meal (McMeal combo)",
            "drink_out_eur":   "domestic draught beer 0.5L at a bar",
            "cocktail_eur":    "club / premium drink (~imported beer x 2.4)",
            "coffee_eur":      "cappuccino",
            "grocery_day_eur": "one person, one day of self-catering groceries",
            "club_entry_eur":  "nightclub cover charge (estimate)",
        },
        "estimated_items": ["cocktail_eur", "club_entry_eur"],
        "sources": [
            "Numbeo public country/city pages, June 2026 (cache/numbeo_raw.json anchors)",
            "Eurostat 2024 Price Level Indices, EU27_2020=100 (long-tail scaling + validation)",
        ],
        "validation": validation,
    },
    "countries": countries,
    "cities":    cities,
}
OUT.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
src_c = Counter(b["source"] for b in countries.values())
print(f"Wrote {OUT}  ({OUT.stat().st_size/1024:.1f} KB)")
print(f"  {len(countries)} countries ({dict(src_c)}), {len(cities)} cities")
print(f"  meals PLI error: {validation['per_item_mae_pct']['meal_mid_eur']}% / {validation['per_item_mae_pct']['meal_cheap_eur']}%")

Wrote cache\costs.json  (20.0 KB)
  42 countries ({'pli_scaled': 18, 'numbeo_direct': 24}), 22 cities
  meals PLI error: 14.7% / 20.2%
